# 2. Modelagem Supervisionada — Previsão de Atrasos de Voos

## Objetivo
Aplicar algoritmos de **classificação** e **regressão** para prever atrasos de voos nos EUA (2015).

- **Classificação**: O voo vai atrasar ≥ 15 minutos? (Sim/Não)  
- **Regressão**: Se atrasar, quantos minutos de atraso?

### Algoritmos comparados
| Tipo | Algoritmos |
|------|-----------|
| Classificação | Random Forest, XGBoost, Logistic Regression |
| Regressão | Random Forest Regressor, XGBoost Regressor |

### Métricas de avaliação
- **Classificação**: Accuracy, Precision, Recall, F1-Score, AUC-ROC
- **Regressão**: MAE, RMSE, R²

## 2.1 Setup e Carregamento dos Dados

Importamos os módulos do projeto e carregamos os dados já limpos com as features engenheiradas (período do dia, estação, feriados, etc.).

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_clean_data
from src.feature_engineering import prepare_features
from src.supervised import (
    run_classification, run_regression, encode_categoricals,
    split_data, FEATURE_COLS
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Carregar dados
flights, airlines, airports = load_clean_data()
flights = prepare_features(flights)
print(f"Dataset: {flights.shape[0]:,} voos × {flights.shape[1]} colunas")
print(f"\nDistribuição da variável alvo (IS_DELAYED):")
print(flights["IS_DELAYED"].value_counts(normalize=True).map("{:.2%}".format))

### Verificação das features

Antes de modelar, vamos confirmar que as features categóricas e numéricas estão presentes e válidas.

In [ ]:
# Features que serão usadas nos modelos
cat_cols = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "PERIODO_DIA", "ESTACAO"]
num_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "SCHEDULED_TIME",
            "IS_WEEKEND", "IS_HOLIDAY", "NEAR_HOLIDAY"]

print("Features categóricas:")
for c in cat_cols:
    print(f"  {c}: {flights[c].nunique()} valores únicos")

print(f"\nFeatures numéricas:")
for c in num_cols:
    print(f"  {c}: min={flights[c].min()}, max={flights[c].max()}, null={flights[c].isnull().sum()}")

print(f"\nTarget (IS_DELAYED): {flights['IS_DELAYED'].dtype}")
print(f"Target (ARRIVAL_DELAY): {flights['ARRIVAL_DELAY'].describe().round(2).to_dict()}")

## 2.2 Classificação — O voo vai atrasar?

**Objetivo**: Prever se um voo terá atraso ≥ 15 minutos na chegada (variável binária `IS_DELAYED`).

**Algoritmos**:
1. **Random Forest** (n_estimators=200, max_depth=15)
2. **XGBoost** (n_estimators=200, max_depth=8, lr=0.1)
3. **Logistic Regression** (max_iter=1000)

**Métricas**: Accuracy, Precision, Recall, F1-Score, AUC-ROC, Matriz de Confusão

> *Nota*: Para datasets > 500K registros, é feita amostragem para viabilizar o treinamento.

In [ ]:
# Executar pipeline de classificação (3 modelos comparados)
clf_results = run_classification(flights)

### Comparação dos Classificadores

Tabela resumo com as métricas de cada algoritmo. O melhor modelo é salvo em `outputs/models/best_classifier.pkl`.

In [ ]:
# Tabela comparativa de classificação (inclui cross-validation)
clf_comparison = pd.DataFrame({
    name: {
        "Accuracy": r["report"]["accuracy"],
        "Precision (delayed)": r["report"]["1"]["precision"],
        "Recall (delayed)": r["report"]["1"]["recall"],
        "F1-Score (delayed)": r["report"]["1"]["f1-score"],
        "AUC-ROC": r["auc"],
        "CV AUC (3-fold)": r["cv_auc_mean"],
        "CV AUC std": r["cv_auc_std"],
    }
    for name, r in clf_results.items()
}).T.round(4)

best_clf = clf_comparison["AUC-ROC"].idxmax()
print(f"Melhor classificador (AUC-ROC): {best_clf}")
print(f"AUC-ROC = {clf_comparison.loc[best_clf, 'AUC-ROC']:.4f}")
print(f"CV AUC-ROC = {clf_comparison.loc[best_clf, 'CV AUC (3-fold)']:.4f} +/- {clf_comparison.loc[best_clf, 'CV AUC std']:.4f}\n")
clf_comparison.style.highlight_max(axis=0, color="lightgreen")

## 2.3 Regressão — Quantos minutos de atraso?

**Objetivo**: Para voos que efetivamente atrasaram (ARRIVAL_DELAY > 0), prever quantos minutos de atraso.

**Algoritmos**:
1. **Random Forest Regressor** (n_estimators=150, max_depth=15)
2. **XGBoost Regressor** (n_estimators=200, max_depth=8, lr=0.1)

**Métricas**: MAE (erro médio absoluto), RMSE (raiz do erro quadrático médio), R² (coeficiente de determinação)

> *Nota*: Filtramos apenas voos com atraso > 0 min para a regressão ser mais realista.

In [ ]:
# Executar pipeline de regressão (2 modelos comparados)
reg_results = run_regression(flights)

### Comparação dos Regressores

Tabela resumo com as métricas de cada algoritmo. O melhor modelo é salvo em `outputs/models/best_regressor.pkl`.

In [ ]:
# Tabela comparativa de regressão (inclui cross-validation)
reg_comparison = pd.DataFrame({
    name: {
        "MAE (min)": r["mae"],
        "RMSE (min)": r["rmse"],
        "R2": r["r2"],
        "CV MAE (3-fold)": r["cv_mae_mean"],
        "CV MAE std": r["cv_mae_std"],
    }
    for name, r in reg_results.items()
}).T.round(4)

best_reg = reg_comparison["MAE (min)"].idxmin()
print(f"Melhor regressor (MAE): {best_reg}")
print(f"MAE = {reg_comparison.loc[best_reg, 'MAE (min)']:.2f} minutos")
print(f"CV MAE = {reg_comparison.loc[best_reg, 'CV MAE (3-fold)']:.2f} +/- {reg_comparison.loc[best_reg, 'CV MAE std']:.2f}\n")
reg_comparison.style.highlight_min(subset=["MAE (min)", "RMSE (min)"], color="lightgreen")\
    .highlight_max(subset=["R2"], color="lightgreen")

## 2.4 Analise Critica — Modelagem Supervisionada

### Interpretacao dos Resultados

**Classificacao**:
- Comparamos 3 algoritmos distintos (Random Forest, XGBoost, Logistic Regression)
- AUC-ROC como criterio principal — lida melhor com desbalanceamento de classes (~73/27)
- **Cross-validation 3-fold** aplicada para validar estabilidade: CV AUC proximo do AUC de teste indica boa generalizacao
- Feature Importance revela que DEP_HOUR, AIRLINE e ORIGIN sao as variaveis mais preditivas
- Matriz de Confusao mostra trade-off entre falsos positivos e falsos negativos
- Features de congestionamento (FLIGHTS_SAME_HOUR_ORIGIN, ROUTE_POPULARITY) melhoram predicao

**Regressao**:
- Filtramos apenas voos com atraso > 0 para evitar vies
- MAE fornece interpretacao direta: "o modelo erra em media X minutos"
- **Cross-validation 3-fold** confirma que o MAE de teste e robusto e nao sofre overfitting
- Grafico Real vs Previsto — pontos proximos a diagonal = bom modelo
- Distribuicao de residuos centrada em zero indica ausencia de vies sistematico
- Para atrasos extremos (>120 min), os regressores tendem a subestimar — esperado pelo efeito dos outliers

### Limitacoes
- Features como clima em tempo real e eventos especiais nao estao disponiveis — incorpora-los melhoraria significativamente os modelos
- Variaveis de detalhe de atraso (AIRLINE_DELAY, WEATHER_DELAY etc.) so existem para voos atrasados — nao podem ser usadas como preditoras (leakage)
- Amostragem (500K para clf, 300K para reg) pode perder padroes de subgrupos minoritarios
- Modelo treinado apenas com dados de 2015 — nao ha garantia de generalizacao temporal
- O desbalanceamento de classes (73/27) pode levar a viés para a classe majoritária (nao atrasado)
- Tecnicas como SMOTE ou class_weight='balanced' poderiam ser exploradas